# Week 7: Fine-tune & Compare – Wakanda Chimobi Ogudu

Load dataset (CSV or Google Drive), fine-tune with LoRA/QLoRA, push to Hugging Face, then compare baseline vs frontier vs fine-tuned (MAE, RMSE, loss).

In [ ]:
import sys
!{sys.executable} -m pip install -q datasets pandas numpy scikit-learn litellm python-dotenv tqdm matplotlib gdown transformers peft bitsandbytes trl accelerate hf_xet

In [ ]:
import os
import re
import random
import csv
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv(override=True)

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

from datasets import Dataset, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

In [ ]:
RANDOM_SEED = 42
MIN_PRICE = 0.5
MAX_PRICE = 999.49
MAX_TEXT_TOTAL = 1200
TRAIN_FRAC = 0.8
VAL_FRAC = 0.1
TEST_FRAC = 0.1
EVAL_SAMPLE_SIZE = 150

# Data: Week 6 CSV (local path or Google Drive share link) or HuggingFace dataset
USE_HF_DATASET = False
DATASET_CSV_PATH = "https://drive.google.com/file/d/1bq3qGfDBv20WXCKPl9mCfE8gIhobBOly/view?usp=sharing"  # local path or https://drive.google.com/... share link
HF_DATASET_NAME = "chimaken/pricer-data"  # if USE_HF_DATASET

# Base model for fine-tuning (small for Colab T4; use 7B+ and QLoRA for A100)
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
USE_QLORA = True  # True = 4-bit QLoRA, False = LoRA only

# Training
OUTPUT_DIR = "./pricer-ft"
NUM_EPOCHS = 2
BATCH_SIZE = 4
LR = 2e-5
LORA_R = 16
LORA_ALPHA = 32
MAX_SEQ_LENGTH = 512

# Hugging Face upload
HF_USER = os.getenv("HF_USER", "your-username")
HF_MODEL_NAME = f"{HF_USER}/pricer-ft-qlora" if USE_QLORA else f"{HF_USER}/pricer-ft-lora"
PUSH_TO_HUB = True

# Frontier model (same as Week 6) for comparison
FRONTIER_MODEL = "openrouter/openai/gpt-4.1-nano"

# OpenRouter API key: in Colab add OPENROUTER_API_KEY in Secrets (🔑); locally use .env
if IN_COLAB:
    try:
        os.environ.setdefault("OPENROUTER_API_KEY", userdata.get("OPENROUTER_API_KEY"))
    except Exception as e:
        print("To use the frontier model: add OPENROUTER_API_KEY in Colab Secrets (🔑).", e)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
@dataclass
class ProductItem:
    title: str
    category: str
    price: float
    full: Optional[str] = None
    summary: Optional[str] = None
    item_id: Optional[int] = None

def _parse_static_row(text: str) -> tuple:
    title = category = None
    for line in text.strip().split("\n"):
        line = line.strip()
        if line.startswith("Title:"): title = line[6:].strip()
        elif line.startswith("Category:"): category = line[9:].strip()
    full = (text or "").strip()[:MAX_TEXT_TOTAL]
    return (title or "Product", category or "Other", full)

def _resolve_csv_path(csv_path: str):
    """If csv_path is a Google Drive URL, download to a temp file. Returns (path, was_drive)."""
    if "drive.google.com" in str(csv_path):
        import gdown
        import tempfile
        with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
            tmp_path = tmp.name
        gdown.download(csv_path, tmp_path, quiet=True, fuzzy=True)
        return tmp_path, True
    path = Path(csv_path)
    if not path.is_absolute(): path = Path.cwd() / path
    return str(path.resolve()), False

def load_from_csv(csv_path: str) -> List[ProductItem]:
    path, was_drive = _resolve_csv_path(csv_path)
    items = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            if len(row) < 2: continue
            text, price_str = row[0], row[1]
            try: price = float(price_str)
            except (ValueError, TypeError): continue
            if not (MIN_PRICE <= price <= MAX_PRICE): continue
            title, category, full = _parse_static_row(text)
            if len(full) < 50: full = f"{title}\n{full}"
            items.append(ProductItem(title=title, category=category, price=price, full=full, summary=full, item_id=i))
    if was_drive and Path(path).exists():
        Path(path).unlink()
    return items

def load_from_hf(dataset_name: str) -> List[ProductItem]:
    from datasets import load_dataset
    ds = load_dataset(dataset_name)
    items = []
    for split in ("train", "test", "validation"):
        if split not in ds: continue
        for i, row in enumerate(ds[split]):
            try: price = float(row.get("price", 0))
            except (ValueError, TypeError): continue
            if not (MIN_PRICE <= price <= MAX_PRICE): continue
            title = (row.get("title") or "Product").strip()
            category = (row.get("category") or "Other").strip()
            full = (row.get("full") or row.get("summary") or row.get("text") or title or "").strip()[:MAX_TEXT_TOTAL]
            if isinstance(full, list): full = " ".join(str(x) for x in full)
            items.append(ProductItem(title=title, category=category, price=price, full=full, summary=full, item_id=len(items)))
    return items

In [ ]:
if USE_HF_DATASET:
    all_items = load_from_hf(HF_DATASET_NAME)
    print(f"Loaded {len(all_items):,} from HF: {HF_DATASET_NAME}")
else:
    all_items = load_from_csv(DATASET_CSV_PATH)
    print(f"Loaded {len(all_items):,} from CSV: {DATASET_CSV_PATH}")

def train_val_test_split(items, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=RANDOM_SEED):
    n = len(items)
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    n_test = n - n_train - n_val
    return [items[i] for i in idx[:n_train]], [items[i] for i in idx[n_train:n_train+n_val]], [items[i] for i in idx[n_train+n_val:]]

train_items, val_items, test_items = train_val_test_split(all_items)
print(f"Train: {len(train_items):,}  Val: {len(val_items):,}  Test: {len(test_items):,}")

In [ ]:
PRICE_PROMPT = "Estimate the price in USD. Reply with only the number."
PRICE_PREFIX = "Price is $"

def item_to_prompt_completion(item: ProductItem) -> dict:
    text = item.summary or item.full or item.title
    prompt = f"{PRICE_PROMPT}\n\n{text}\n\n{PRICE_PREFIX}"
    completion = f"{item.price:.2f}"
    return {"prompt": prompt, "completion": completion}

def format_for_sft(items: List[ProductItem]) -> List[dict]:
    return [item_to_prompt_completion(it) for it in items]

train_sft = format_for_sft(train_items)
val_sft = format_for_sft(val_items)
train_ds = Dataset.from_list(train_sft)
val_ds = Dataset.from_list(val_sft)
print(f"SFT train: {len(train_ds)}, val: {len(val_ds)}")

In [ ]:
def push_dataset_to_hub(name: str):
    from huggingface_hub import login
    token = userdata.get("HF_TOKEN") if IN_COLAB else os.getenv("HF_TOKEN")
    if token: login(token, add_to_git_credential=True)
    ds_dict = DatasetDict({"train": train_ds, "validation": val_ds})
    ds_dict.push_to_hub(name)
    print(f"Pushed to {name}")

# Uncomment to push:
push_dataset_to_hub(f"{HF_USER}/pricer-sft-data")

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import login
    hf_token = userdata.get("HF_TOKEN") if IN_COLAB else os.getenv("HF_TOKEN")
    if hf_token: login(hf_token, add_to_git_credential=True)
    else: print("Set HF_TOKEN in .env or Colab Secrets to push to Hub.")

def get_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    if USE_QLORA:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype="float16", bnb_4bit_quant_type="nf4")
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto", trust_remote_code=True)
    else:
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto", trust_remote_code=True)
    lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM)
    model = get_peft_model(model, lora)
    return model, tokenizer

In [ ]:
model, tokenizer = get_model_and_tokenizer()

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HF_MODEL_NAME,
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=sft_config,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
if PUSH_TO_HUB:
    trainer.push_to_hub()
print(f"Done. Model in {OUTPUT_DIR}" + (f", pushed to {HF_MODEL_NAME}" if PUSH_TO_HUB else ""))

In [ ]:
def parse_price_from_response(response: str) -> float:
    if response is None: return 0.0
    s = str(response).replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return max(0.0, float(m.group())) if m else 0.0

def compute_metrics(actual: List[float], pred: List[float]) -> dict:
    a, p = np.asarray(actual, dtype=float), np.asarray(pred, dtype=float)
    return {"mae": np.mean(np.abs(a - p)), "rmse": np.sqrt(mean_squared_error(a, p)), "mse": mean_squared_error(a, p), "r2": r2_score(a, p)}

def run_benchmark(predictor, test_data, sample_size=EVAL_SAMPLE_SIZE, return_arrays=False):
    size = min(sample_size, len(test_data))
    subset = random.Random(RANDOM_SEED).sample(test_data, size)
    actuals, preds = [], []
    for item in tqdm(subset, desc="Evaluating"):
        out = predictor(item)
        preds.append(parse_price_from_response(out) if isinstance(out, str) else float(out))
        actuals.append(item.price)
    metrics = compute_metrics(actuals, preds)
    return (metrics, actuals, preds) if return_arrays else metrics

In [ ]:
mean_train = np.mean([p.price for p in train_items])
def baseline_pricer(item): return mean_train

baseline_metrics, eval_actuals, eval_baseline = run_benchmark(baseline_pricer, test_items, return_arrays=True)
print("Baseline (mean):", baseline_metrics)

In [ ]:
try:
    from litellm import completion
except ImportError:
    completion = None

def frontier_pricer(item):
    if completion is None: return "0"
    text = item.summary or item.full or item.title
    msg = [{"role": "user", "content": f"Estimate the price in USD. Reply with only the number.\n\n{text}"}]
    api_key = os.getenv("OPENROUTER_API_KEY")
    try: return completion(model=FRONTIER_MODEL, messages=msg, api_key=api_key).choices[0].message.content or "0"
    except Exception as e: print(f"API error: {e}"); return "0"

frontier_metrics, _, eval_frontier = run_benchmark(frontier_pricer, test_items, return_arrays=True)
print(f"Frontier ({FRONTIER_MODEL}):", frontier_metrics)

In [ ]:
from peft import PeftModel

FT_MODEL_PATH = OUTPUT_DIR  # or HF_MODEL_NAME to load from Hub
ft_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if USE_QLORA:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype="float16", bnb_4bit_quant_type="nf4")
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
else:
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")
ft_model = PeftModel.from_pretrained(base, FT_MODEL_PATH)
ft_model.eval()

def finetuned_pricer(item):
    text = item.summary or item.full or item.title
    prompt = f"{PRICE_PROMPT}\n\n{text}\n\n{PRICE_PREFIX}"
    inputs = ft_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(ft_model.device)
    out = ft_model.generate(**inputs, max_new_tokens=16, do_sample=False, pad_token_id=ft_tokenizer.eos_token_id)
    reply = ft_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return reply

finetuned_metrics, _, eval_finetuned = run_benchmark(finetuned_pricer, test_items, return_arrays=True)
print("Fine-tuned:", finetuned_metrics)

## 8. Compare all three models

In [ ]:
comparison = pd.DataFrame([
    {"Model": "Baseline (mean)", "MAE": baseline_metrics["mae"], "RMSE": baseline_metrics["rmse"], "Loss (MSE)": baseline_metrics["mse"], "R²": baseline_metrics["r2"]},
    {"Model": f"Frontier ({FRONTIER_MODEL.split('/')[-1]})", "MAE": frontier_metrics["mae"], "RMSE": frontier_metrics["rmse"], "Loss (MSE)": frontier_metrics["mse"], "R²": frontier_metrics["r2"]},
    {"Model": "Fine-tuned (LoRA/QLoRA)", "MAE": finetuned_metrics["mae"], "RMSE": finetuned_metrics["rmse"], "Loss (MSE)": finetuned_metrics["mse"], "R²": finetuned_metrics["r2"]},
])
print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax1, ax2 = axes[0], axes[1]
max_p = max(max(eval_actuals), max(eval_baseline), max(eval_frontier), max(eval_finetuned))
ax1.scatter(eval_actuals, eval_baseline, alpha=0.5, s=20, label="Baseline", color="C1")
ax1.scatter(eval_actuals, eval_frontier, alpha=0.5, s=20, label="Frontier", color="C0")
ax1.scatter(eval_actuals, eval_finetuned, alpha=0.5, s=20, label="Fine-tuned", color="C2")
ax1.plot([0, max_p], [0, max_p], "k--", alpha=0.7, label="Perfect")
ax1.set_xlabel("Actual price ($)"); ax1.set_ylabel("Predicted price ($)"); ax1.set_title("Actual vs Predicted"); ax1.legend(); ax1.set_xlim(0, max_p*1.02); ax1.set_ylim(0, max_p*1.02)

models = ["Baseline", "Frontier", "Fine-tuned"]
mae_vals = [baseline_metrics["mae"], frontier_metrics["mae"], finetuned_metrics["mae"]]
rmse_vals = [baseline_metrics["rmse"], frontier_metrics["rmse"], finetuned_metrics["rmse"]]
x = np.arange(3); w = 0.35
ax2.bar(x - w/2, mae_vals, w, label="MAE ($)", color="C2"); ax2.bar(x + w/2, rmse_vals, w, label="RMSE ($)", color="C3")
ax2.set_ylabel("Error ($)"); ax2.set_title("MAE & RMSE by model"); ax2.set_xticks(x); ax2.set_xticklabels(models); ax2.legend()
plt.tight_layout(); plt.show()